Universidade do Vale do Itajaí<br>
Escola do Mar, Ciência e Tecnologia<br>
Engenharia de Computação - Processamento Digital de Sinais: Imagens

### Tutoriais da OpenCV

- https://docs.opencv.org/master/d9/df8/tutorial_root.html
- https://www.geeksforgeeks.org/opencv-python-tutorial/

# Algoritmos de Segmentação de Imagens - Visão Prática: Otsu, Canny, Watershed, SLIC e K-means

Demonstração interno de cada algoritmo sem depender de bibliotecas como OpenCV ou scikit-image.


In [1]:
import numpy as np
from heapq import heappush, heappop # para usar com o algoritmo Watershed
import cv2

## 1. Otsu

O método de Otsu encontra automaticamente um limiar global que maximiza a variância entre duas classes (fundo e objeto).

In [2]:

def otsu_threshold(image):
    image = image.astype(np.uint8)
    hist = np.bincount(image.ravel(), minlength=256)
    prob = hist / image.size

    cumulative_prob = np.cumsum(prob)
    cumulative_mean = np.cumsum(prob * np.arange(256))
    global_mean = cumulative_mean[-1]

    numerator = (global_mean * cumulative_prob - cumulative_mean) ** 2
    denominator = cumulative_prob * (1 - cumulative_prob)
    denominator = np.where(denominator == 0, 1e-10, denominator)

    sigma_b2 = numerator / denominator
    return int(np.argmax(sigma_b2))

def apply_threshold(image, threshold):
    return (image >= threshold).astype(np.uint8) * 255


In [3]:
# Versão mais from scratch

def otsu_threshold_manual(image):
    hist = np.bincount(image.ravel(), minlength=256)
    total = image.size

    sum_total = np.sum(np.arange(256) * hist)

    sum_b = 0
    weight_b = 0
    max_variance = 0
    threshold = 0

    for t in range(256):
        weight_b += hist[t]
        if weight_b == 0:
            continue

        weight_f = total - weight_b
        if weight_f == 0:
            break

        sum_b += t * hist[t]

        mean_b = sum_b / weight_b
        mean_f = (sum_total - sum_b) / weight_f

        # Variância entre classes
        var_between = weight_b * weight_f * (mean_b - mean_f) ** 2

        if var_between > max_variance:
            max_variance = var_between
            threshold = t

    return threshold

## Exemplo de Uso

Dicas:
- Funciona melhor quando a imagem tem bimodalidade (dois picos no histograma)
- Não precisa de parâmetros, o que torna fácil a aplicação (não a implementação)

In [8]:
# Criando imagem sintética simples
image = np.zeros((100, 100), dtype=np.uint8)
image[:, :50] = 50     # fundo escuro
image[:, 50:] = 200    # objeto claro

# Calcula threshold
t = otsu_threshold(image)
print("Threshold ótimo:", t)

# Binariza
binary = apply_threshold(image, t)

cv2.imshow('in', image)
cv2.waitKey(0)
cv2.destroyAllWindows()

cv2.imshow('in', binary)
cv2.waitKey(0)
cv2.destroyAllWindows()

Threshold ótimo: 50


## 2. Canny

Pipeline: suavização Gaussiana, Sobel, Non-Maximum Suppression, double threshold e hysteresis.

In [10]:

def convolve2d(image, kernel):
    h, w = image.shape
    kh, kw = kernel.shape
    ph, pw = kh // 2, kw // 2
    padded = np.pad(image, ((ph, ph), (pw, pw)), mode='constant')
    out = np.zeros((h, w), dtype=float)

    for i in range(h):
        for j in range(w):
            out[i, j] = np.sum(padded[i:i+kh, j:j+kw] * kernel)
    return out

def gaussian_kernel(size=5, sigma=1.0):
    ax = np.arange(-(size // 2), size // 2 + 1)
    xx, yy = np.meshgrid(ax, ax)
    kernel = np.exp(-(xx**2 + yy**2)/(2*sigma**2))
    return kernel / np.sum(kernel)

def sobel_filters(image):
    Kx = np.array([[-1,0,1],[-2,0,2],[-1,0,1]])
    Ky = np.array([[1,2,1],[0,0,0],[-1,-2,-1]])
    Gx = convolve2d(image, Kx)
    Gy = convolve2d(image, Ky)
    magnitude = np.hypot(Gx, Gy)
    magnitude = magnitude / (magnitude.max() + 1e-10) * 255
    direction = np.arctan2(Gy, Gx)
    return magnitude, direction

def non_max_suppression(magnitude, direction):
    h, w = magnitude.shape
    out = np.zeros((h, w), dtype=np.float32)
    angle = np.degrees(direction)
    angle[angle < 0] += 180

    for i in range(1, h-1):
        for j in range(1, w-1):
            q = r = 0
            a = angle[i, j]

            if (0 <= a < 22.5) or (157.5 <= a <= 180):
                q, r = magnitude[i, j+1], magnitude[i, j-1]
            elif 22.5 <= a < 67.5:
                q, r = magnitude[i+1, j-1], magnitude[i-1, j+1]
            elif 67.5 <= a < 112.5:
                q, r = magnitude[i+1, j], magnitude[i-1, j]
            else:
                q, r = magnitude[i-1, j-1], magnitude[i+1, j+1]

            if magnitude[i, j] >= q and magnitude[i, j] >= r:
                out[i, j] = magnitude[i, j]
    return out

def double_threshold(image, T1=30, T2=100):
    weak, strong = 75, 255
    result = np.zeros(image.shape, dtype=np.uint8)
    result[image >= T2] = strong
    result[(image >= T1) & (image < T2)] = weak
    return result, weak, strong

def hysteresis(image, weak=75, strong=255):
    h, w = image.shape
    out = image.copy()
    for i in range(1, h-1):
        for j in range(1, w-1):
            if out[i, j] == weak:
                if np.any(out[i-1:i+2, j-1:j+2] == strong):
                    out[i, j] = strong
                else:
                    out[i, j] = 0
    return out

def canny(image, sigma=1.0, T1=30, T2=100):
    smoothed = convolve2d(image, gaussian_kernel(5, sigma))
    magnitude, direction = sobel_filters(smoothed)
    nms = non_max_suppression(magnitude, direction)
    thresh, weak, strong = double_threshold(nms, T1, T2)
    return hysteresis(thresh, weak, strong)


## Exemplo de Uso

Dica:
- Essa implementação é didática, não otimizada
- Gargalos principais:
    - loops em Python (NMS + histerese)
- Regra comum: T1 ≈ 0.3 * T2
  - Você pode calcular T1/T2 automaticamente usando Otsu

In [12]:
# Imagem sintética
image = np.zeros((100, 100), dtype=np.float32)
image[:, 50:] = 255  # borda vertical

edges = canny(image)

print(edges)

edges_t1t2 = canny(image, T1=30, T2=100)

cv2.imshow('in', image)
cv2.waitKey(0)
cv2.destroyAllWindows()

cv2.imshow('out1', edges)
cv2.waitKey(0)
cv2.destroyAllWindows()

cv2.imshow('out2', edges_t1t2)
cv2.waitKey(0)
cv2.destroyAllWindows()

[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]


## 3. Watershed

Propaga rótulos a partir de marcadores usando uma fila de prioridade.

Dica para o entendimento:
    - Imagine a imagem como um relevo:
      - pixels claros = regiões altas
      - pixels escuros = vales
    - A água “sobe” a partir de marcadores:
      - Regiões vão crescendo
      - Quando duas regiões se encontram, cria-se uma borda (watershed)

Implementação baseada em:
    - Imagem de entrada (geralmente gradiente)
    - Marcadores iniciais (labels)
    - Flooding controlado por prioridade (menor intensidade primeiro)

In [2]:

def get_neighbors(i, j, h, w):
    for di in (-1, 0, 1):
        for dj in (-1, 0, 1):
            if di == 0 and dj == 0:
                continue
            ni, nj = i + di, j + dj
            if 0 <= ni < h and 0 <= nj < w:
                yield ni, nj

def watershed(image, markers):
    h, w = image.shape
    labels = markers.copy()
    pq = []

    for i in range(h):
        for j in range(w):
            if labels[i, j] > 0:
                heappush(pq, (image[i, j], i, j))

    while pq:
        _, i, j = heappop(pq)
        for ni, nj in get_neighbors(i, j, h, w):
            if labels[ni, nj] == 0:
                labels[ni, nj] = labels[i, j]
                heappush(pq, (image[ni, nj], ni, nj))
            elif labels[ni, nj] != labels[i, j]:
                labels[i, j] = -1
    return labels


In [3]:
# Imagem simples (gradiente)
image = np.array([
    [10, 10, 10, 50, 50],
    [10, 10, 10, 50, 50],
    [10, 10, 10, 50, 50],
    [80, 80, 80, 20, 20],
    [80, 80, 80, 20, 20],
], dtype=np.float32)

# Marcadores:
# 1 = objeto 1
# 2 = objeto 2
markers = np.zeros_like(image, dtype=np.int32)
markers[0, 0] = 1
markers[4, 4] = 2

labels = watershed(image, markers)

print(labels)

cv2.imshow('in', image)
cv2.waitKey(0)
cv2.destroyAllWindows()


[[ 1  1  1  1  1]
 [ 1  1  1  1  1]
 [ 1  1  1 -1 -1]
 [ 1  1 -1 -1 -1]
 [ 1 -1 -1 -1 -1]]


## 4. SLIC

Agrupa pixels considerando cor/intensidade e posição espacial.

O SLIC funciona como um k-means espacial + cor:
    - Cada cluster tem:
      - posição (x, y)
      - cor (intensidade ou RGB)
    - A distância mistura:
      - distância espacial
      - distância de cor
    - Resultado: regiões compactas e coerentes


$$ D_{SLIC} = \sqrt{d_c^2 + \left(\frac{m}{S}\right)^2 d_s^2}$$

$$
d_c =
\sqrt{
\sum_{c=1}^{C}
\left(I_i^{(c)} - I_k^{(c)}\right)^2
}
$$

$$
d_s =
\sqrt{
(x_i - x_k)^2 +
(y_i - y_k)^2
}
$$

$$
S =
\sqrt{
\frac{N}{K}
}
$$

$ d_c $ = distância de cor

$ d_s $ = distância espacial

$ S $ = tamanho esperado do superpixel

$ m $ = compactação


In [ ]:

def initialize_clusters(image, S):
    h, w = image.shape[:2]
    clusters = []
    for y in range(S // 2, h, S):
        for x in range(S // 2, w, S):
            clusters.append([float(y), float(x), image[y, x].astype(float) if hasattr(image[y, x], 'astype') else float(image[y, x])])
    return clusters

def compute_slic_distance(y, x, cluster, image, S, m):
    yc, xc, ic = cluster
    ds2 = (y - yc)**2 + (x - xc)**2
    pixel = image[y, x]
    if image.ndim == 2:
        dc2 = (float(pixel) - float(ic))**2
    else:
        dc2 = np.sum((pixel.astype(float) - np.asarray(ic, dtype=float))**2)
    return np.sqrt(dc2 + (m / S)**2 * ds2)

def slic(image, num_superpixels=100, m=10, max_iter=10):
    h, w = image.shape[:2]
    S = max(1, int(np.sqrt((h * w) / num_superpixels)))
    clusters = initialize_clusters(image, S)
    K = len(clusters)

    labels = -np.ones((h, w), dtype=np.int32)
    distances = np.full((h, w), np.inf)

    for _ in range(max_iter):
        for k, cluster in enumerate(clusters):
            yc, xc, _ = cluster
            y0, y1 = int(max(yc - S, 0)), int(min(yc + S, h))
            x0, x1 = int(max(xc - S, 0)), int(min(xc + S, w))

            for y in range(y0, y1):
                for x in range(x0, x1):
                    d = compute_slic_distance(y, x, cluster, image, S, m)
                    if d < distances[y, x]:
                        distances[y, x] = d
                        labels[y, x] = k

        new_clusters = []
        for k in range(K):
            ys, xs = np.where(labels == k)
            if len(ys) == 0:
                new_clusters.append(clusters[k])
                continue
            yc, xc = np.mean(ys), np.mean(xs)
            if image.ndim == 2:
                ic = np.mean(image[ys, xs])
            else:
                ic = np.mean(image[ys, xs], axis=0)
            new_clusters.append([yc, xc, ic])

        clusters = new_clusters
        distances.fill(np.inf)

    return labels


## Exemplo de Uso

In [ ]:
# imagem simples
image = np.random.randint(0, 255, (100, 100), dtype=np.uint8)

labels = slic(image, num_superpixels=50, m=10)

print(labels.shape)

cv2.imshow('in', image)
cv2.waitKey(0)
cv2.destroyAllWindows()

## 5. K-Means para Segmentação

Agrupa pixels com base na intensidade (tons de cinza) ou cor (RGB).

In [ ]:

def kmeans_segmentation(image, k=3, max_iter=100, tol=1e-4, random_state=None):
    rng = np.random.default_rng(random_state)

    if image.ndim == 2:
        pixels = image.reshape(-1, 1).astype(np.float64)
        original_shape = image.shape
        channels = 1
    else:
        h, w, c = image.shape
        pixels = image.reshape(-1, c).astype(np.float64)
        original_shape = (h, w)
        channels = c

    n_pixels = pixels.shape[0]
    indices = rng.choice(n_pixels, size=k, replace=False)
    centroids = pixels[indices].copy()

    labels = np.zeros(n_pixels, dtype=np.int32)

    for _ in range(max_iter):
        # Atribuição
        distances = np.sqrt(((pixels[:, None, :] - centroids[None, :, :]) ** 2).sum(axis=2))
        new_labels = np.argmin(distances, axis=1)

        # Atualização
        new_centroids = centroids.copy()
        for cluster in range(k):
            cluster_pixels = pixels[new_labels == cluster]
            if len(cluster_pixels) > 0:
                new_centroids[cluster] = cluster_pixels.mean(axis=0)
            else:
                new_centroids[cluster] = pixels[rng.integers(0, n_pixels)]

        # Critério de convergência
        shift = np.linalg.norm(new_centroids - centroids)
        labels = new_labels
        centroids = new_centroids

        if shift < tol:
            break

    segmented = centroids[labels]

    if channels == 1:
        segmented = segmented.reshape(original_shape)
    else:
        segmented = segmented.reshape(*original_shape, channels)

    return segmented.astype(image.dtype), labels.reshape(original_shape), centroids


## Exemplo de uso

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Criando uma imagem sintética em tons de cinza com 3 regiões
image = np.zeros((150, 150), dtype=np.uint8)

# Região 1: escura
image[:, :50] = 40

# Região 2: média
image[:, 50:100] = 130

# Região 3: clara
image[:, 100:] = 220

# Adicionando ruído
noise = np.random.randint(-20, 21, size=image.shape)
image = np.clip(image.astype(np.int32) + noise, 0, 255).astype(np.uint8)

# Aplicando K-Means

segmented, labels, centroids = kmeans_segmentation(image, k=3, max_iter=100, random_state=42)

# Ordenando centróides para facilitar interpretação
print("Centróides encontrados:")
print(np.sort(centroids.flatten()))


In [ ]:
#Plotar a segmentação
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

axes[0].imshow(image, cmap='gray')
axes[0].set_title("Imagem Original")
axes[0].axis("off")

axes[1].imshow(labels, cmap='jet')
axes[1].set_title("Mapa de Labels")
axes[1].axis("off")

axes[2].imshow(segmented, cmap='gray')
axes[2].set_title("Imagem Segmentada")
axes[2].axis("off")

plt.tight_layout()
plt.show()

## Dicas do K-means

```python
segmented, labels, centroids = kmeans_segmentation(image, k=4, random_state=42)
```

- `segmented`: imagem reconstruída com as cores dos centróides.
- `labels`: mapa de rótulos de cada pixel.
- `centroids`: valores médios de cada cluster.
